In [10]:
import requests

def get_uploaded_video_urls(api_key, channel_id):
    # Step 1: Get the uploads playlist ID
    url = "https://www.googleapis.com/youtube/v3/channels"
    params = {
        "part": "contentDetails",
        "id": channel_id,
        "key": api_key
    }
    r = requests.get(url, params=params).json()

    uploads_playlist = r["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]

    # Step 2: Fetch all videos from the uploads playlist
    video_urls = []
    next_page = None

    while True:
        playlist_url = "https://www.googleapis.com/youtube/v3/playlistItems"
        playlist_params = {
            "part": "contentDetails",
            "playlistId": uploads_playlist,
            "maxResults": 50,
            "key": api_key
        }
        if next_page:
            playlist_params["pageToken"] = next_page

        playlist_response = requests.get(playlist_url, params=playlist_params).json()

        for item in playlist_response["items"]:
            video_id = item["contentDetails"]["videoId"]
            video_urls.append(f"https://www.youtube.com/watch?v={video_id}")

        next_page = playlist_response.get("nextPageToken")
        if not next_page:
            break

    return video_urls


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()  # reads .env file

API_KEY = os.getenv("YT_API_KEY")
CHANNEL_ID = os.getenv("YT_CHANNEL_ID")

urls = get_uploaded_video_urls(API_KEY, CHANNEL_ID)
print(urls)


In [8]:
import subprocess
import re
import pytesseract
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

from moviepy.editor import VideoFileClip
from PIL import Image
import numpy as np
from collections import defaultdict
import os
import uuid

def download_video(url):
    """Downloads a YouTube video using yt-dlp and returns the file path."""
    outname = f"video_{uuid.uuid4()}.mp4"
    cmd = [
        "yt-dlp",
        "-f", "mp4",
        "-o", outname,
        url
    ]
    subprocess.run(cmd, check=True)
    return outname


def extract_summary_from_video(url):
    """Downloads a YouTube video, extracts the last 10 seconds,
       performs OCR, and returns parsed player stats."""
    
    # === Step 1: Download video ===
    filepath = download_video(url)

    # === Step 2: Load video and extract last frame ===
    clip = VideoFileClip(filepath)
    end_time = clip.duration
    start_time = max(0, end_time - 10)

    # Take a frame from the middle of the summary section
    frame = clip.get_frame(start_time + 5)
    img = Image.fromarray(frame)

    # === Step 3: OCR ===
    raw_text = pytesseract.image_to_string(img)

    # === Step 4: Parse summary lines ===
    stats = {}
    pattern = r"(.+?):\s*(\d+)\s*goal[s]?,\s*(\d+)\s*assist[s]?"

    for line in raw_text.split("\n"):
        match = re.search(pattern, line.strip())
        if match:
            player = match.group(1).strip()
            goals = int(match.group(2))
            assists = int(match.group(3))
            stats[player] = {"goals": goals, "assists": assists}

    # Cleanup
    clip.close()
    os.remove(filepath)

    return stats


def aggregate_player_stats(video_urls):
    """Aggregates stats across multiple YouTube videos."""
    totals = defaultdict(lambda: {"goals": 0, "assists": 0})

    for url in video_urls:
        summary = extract_summary_from_video(url)
        for player, s in summary.items():
            totals[player]["goals"] += s["goals"]
            totals[player]["assists"] += s["assists"]

    return dict(totals)


In [9]:
videos = [
    "https://www.youtube.com/watch?v=mVdrsEQ02Gk",
]


totals = aggregate_player_stats(videos)
print(totals)


{'Stu': {'goals': 4, 'assists': 4}, 'Ben': {'goals': 4, 'assists': 2}, 'George': {'goals': 4, 'assists': 0}, 'Gerald': {'goals': 3, 'assists': 3}, 'Mat': {'goals': 2, 'assists': 2}, '‘Sam': {'goals': 1, 'assists': 1}, 'Yaw': {'goals': 0, 'assists': 1}, 'Theo': {'goals': 0, 'assists': 1}, 'Luke': {'goals': 0, 'assists': 1}, '‘JC': {'goals': 0, 'assists': 1}, 'Henry': {'goals': 0, 'assists': 1}}
